# Biopython으로 인플루엔자 서열 분석해보기
## 개요
한타바이러스 하다가 생각난건데, NCBI에 최근에 유행했던 인플루엔자 데이터도 있는거 아닌가 해서 하게 됐습니다. 뭐 이것도 그렇게 복잡하지는 않아요. 근데 인플루엔자는 종류가 워낙 많다는게 특징... 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm # 넌 뭐냐 

# BioPython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import AlignInfo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.BaseTree import BranchColor

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import Counter
import re

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔바른펜(본인 기본 고딕 싫어함)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일 
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요(which 치면 나옴)

In [ ]:
# 인플루엔자 H3N2 서열 가져오기
# 쿼리 조건: 인플루엔자 A, 특정 아형, HA 유전자, 최근 1년(2025), 호스트가 사람 
query = f"Influenza A virus AND H3N2 AND HA[Gene Name] AND 2025[PDAT] AND Homo sapiens[Host]" # 사람독감 찾으려면... 

# 1. ID 리스트 가져오기
handle = Entrez.esearch(db="nucleotide", term=query, retmax=300)
record = Entrez.read(handle)
id_list = record["IdList"]
handle.close()

# 2. 실제 서열 데이터 가져오기 (FASTA 형식)
fetch_handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="fasta", retmode="text")
sequences = list(SeqIO.parse(fetch_handle, "fasta"))
fetch_handle.close()

# 3. 저장 
with open("influenza_h3n2.fasta", "w") as f:
    SeqIO.write(sequences, f, "fasta")

print(f"성공적으로 {len(sequences)}개의 서열을 가져왔습니다.")
print("----------")

for record in sequences[:3]:
    print(f"ID: {record.id}")
    print(f"Description: {record.description}")
    print(f"Length: {len(record.seq)} bp\n")

In [ ]:
# H3N2만 필터 (헤더에 H3N2가 들어간다고 가정)
h3n2_records = [
    record for record in alignment
    if "H3N2" in record.id or "H3N2" in record.description
]

alignment = MultipleSeqAlignment(h3n2_records)

print(f"H3N2 sequences: {len(alignment)}")

In [ ]:
# 사전통계-어느 지역 데이터를 얼마나 긁어왔는가? 
locations = []
for record in sequences:
    # Description에서 괄호 안의 지역 정보 추출 (예: A/Shanghai/...)
    match = re.search(r'A/([^/]+)/', record.description)
    if match:
        locations.append(match.group(1))

# 지역별 빈도수 확인
location_counts = Counter(locations)
print("--- 수집된 데이터 지역 분포 ---")
for loc, count in location_counts.most_common():
    print(f"{loc}: {count}개")

In [ ]:
# 지역별 라벨링(함수)
def clean_flu_labels(sequences):
    for record in sequences:
        # 1. 지역 추출 (A/지역/...)
        loc_match = re.search(r'A/([^/]+)/', record.description)
        location = loc_match.group(1) if loc_match else "Unknown"
        
        # 2. 연도 추출 (4자리 숫자)
        year_match = re.search(r'/(\d{4})', record.description)
        year = year_match.group(1) if year_match else "XXXX"
        
        # 3. 새로운 ID 생성 (예: 2023_Shanghai_H3N2)
        # 나중에 트리에 그릴 때 가독성을 위해 짧고 강렬하게!
        record.id = f"{year}_{location}"
        record.description = record.id # 설명도 통일
    return sequences

# 라벨 정리 실행
labeled_sequences = clean_flu_labels(sequences)

# 확인
for r in labeled_sequences[:5]:
    print(r.id)

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try: 
    result = subprocess.run([muscle_exe, "-align", "influenza_h3n2.fasta", "-output", "influenza_h3n2_muscle_aligned.fasta"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e: 
    print(f"MSA failed: {e}")
finally:
    alignment = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta")

# 오래 걸리니까 이거 돌려놓고 잠깐 바람 쐬고 오십쇼 

In [ ]:
print("====== MSA Result ======")
alignment = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta") # FASTA 니네 확장자가 몇개냐... 

for record in alignment:
    print(f"{record.id[:10]:<15} : {record.seq[:100]}")

In [ ]:
# 1. Consensus 서열 계산 (표준형 추출)
summary_align = AlignInfo.SummaryInfo(alignment)
consensus = summary_align.dumb_consensus(threshold=0.5)

# 2. 분석 구간 설정 (559번 염기 주변 40bp)
start, end = 540, 580
subset_align = alignment[:, start:end]
names = [rec.id.split('_')[1] if '_' in rec.id else rec.id for rec in subset_align]

# 3. 데이터 수치화 (Consensus와 같으면 0, 다르면 1)
data = []
for record in subset_align:
    row = [1 if record.seq[i] != consensus[start+i] else 0 for i in range(len(record.seq))]
    data.append(row)

# 4. 시각화
fig, ax = plt.subplots(figsize=(15, len(subset_align) * 0.3))
im = ax.imshow(data, aspect='auto', cmap='Reds', interpolation='nearest')

# 축 설정
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xticks(range(0, end-start, 5))
ax.set_xticklabels(range(start, end, 5))
ax.set_title(f"H3N2 HA Gene Mutation Hotspots (Pos: {start}-{end})", fontsize=15)
ax.set_xlabel("Nucleotide Position")

# 559번 위치 표시
target_idx = 558 - start
ax.axvline(x=target_idx, color='blue', linestyle='--', alpha=0.5)
ax.text(target_idx, len(names) + 0.5, 'Pos 559', color='blue', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("MSA_Mutation_Highlight.png", dpi=300)
plt.show()

In [ ]:
# MSA 요약
# --- 설정값 ---
TOP_N = 30  # 상위 30개만 출력 (화질 확보용)
# 559가 아니라면, 분석하고자 하는 실제 위치로 변경하세요 (예: 226)
REAL_TARGET_POS = 1734
start, end = REAL_TARGET_POS - 20, REAL_TARGET_POS + 20

# 1. 데이터 슬라이싱 (상위 N개만)
subset_align = alignment[:TOP_N, start:end]
names = [rec.id.split('_')[1] if '_' in rec.id else rec.id for rec in subset_align]

# 2. 데이터 수치화
data = []
for record in subset_align:
    row = [1 if record.seq[i] != consensus[start+i] else 0 for i in range(len(record.seq))]
    data.append(row)

# 3. 시각화 (DPI를 높이고 텍스트 겹침 방지)
fig, ax = plt.subplots(figsize=(15, 8), dpi=200) # 높이를 줄여서 더 짱짱하게 만듦
im = ax.imshow(data, aspect='auto', cmap='Reds', interpolation='nearest')

# 축 설정
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=11)
ax.set_xticks(range(0, end-start, 5))
ax.set_xticklabels(range(start, end, 5))

# 제목과 여백
ax.set_title(f"H3N2 HA Gene Focus: Mutations around Pos {REAL_TARGET_POS}", fontsize=16, pad=25)
ax.set_xlabel("Nucleotide Position", fontsize=12)

# --- 타겟 라인 표시 ---
target_idx = REAL_TARGET_POS - 1 - start
ax.axvline(x=target_idx, color='blue', linestyle='--', alpha=0.6, linewidth=2)

# 글자를 제목과 겹치지 않게 '그래프 내부 하단'이나 '축 바로 위'에 배치
ax.text(target_idx, -0.7, f'Pos {REAL_TARGET_POS}', color='blue', 
        ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig("H3N2_Top_Mutation_Map.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 엔트로피 점수 도출
def get_top_variable_sites(alignment, top_n=5):
    length = alignment.get_alignment_length()
    variability = []
    for i in range(length):
        column = alignment[:, i]
        # 가장 많이 등장하는 염기 비율 계산
        most_common_ratio = column.count(max(set(column), key=column.count)) / len(column)
        variability.append((i, 1 - most_common_ratio))
    
    # 변이율이 높은 순으로 정렬
    return sorted(variability, key=lambda x: x[1], reverse=True)[:top_n]

top_sites = get_top_variable_sites(alignment)
print("--- 변이가 집중된 주요 포지션 ---")
for pos, score in top_sites:
    print(f"Position {pos}: 변이율 {score*100:.1f}%")

In [ ]:
align = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta")
length = align.get_alignment_length()
n_seqs = len(align)

entropy_list = []
for i in range(length):
    col = align[:, i]
    # 각 자리의 염기 조성 확인
    chars = set(col)
    probs = [col.count(c) / n_seqs for c in chars]
    # 샤논 엔트로피 계산 (변이의 척도)
    entropy = -sum(p * np.log2(p) for p in probs if p > 0)
    entropy_list.append(entropy)

plt.figure(figsize=(20, 6))
plt.plot(entropy_list, color='darkblue', linewidth=1)
plt.fill_between(range(length), entropy_list, color='skyblue', alpha=0.4)
plt.title("H3N2 HA Sequence Entropy (Prediction Map)", fontsize=15)
plt.xlabel("Amino Acid Position")
plt.ylabel("Entropy (Mutation Intensity)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 트! 리! 
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

terms = tree.get_terminals()
x_limit = max([tree.distance(t) for t in terms])
fig = plt.figure(figsize=(20, 60), dpi=150) # 난 해상도 설정도 될 줄 몰랐고... 
ax = fig.add_subplot(1, 1, 1)

for clade in tree.get_terminals():
    original_name = str(clade.name)
    if '_' in original_name:
        parts = original_name.split('_')
        clade.name = f"[{parts[0]}] {parts[1]} ({original_name})"
    else:
        clade.name = original_name

Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda x: "", show_confidence=False)

# 내가 진짜 이것때문에 제미나이랑 급나 씨름했는데 색깔이 안바껴요. 
# 이름도 몇번이나 했는데 ID만 줄창떠요. 아오. 
for i, node in enumerate(terms):
    y_pos = i + 1  # 가지의 y축 위치
    x_pos = tree.distance(node) # 가지가 끝나는 x축 위치
    
    orig_name = str(node.name)
    # 이름 가공: [연도] 지역 (ID)
    if '_' in orig_name:
        p = orig_name.split('_')
        # 혹시 이미 가공된 이름이라면 중복 방지
        display_text = f"  ◀ [{p[0]}] {p[1]}" if '[' not in orig_name else f"  ◀ {orig_name}"
    else:
        display_text = f"  ◀ {orig_name}"
    
    # 가지 끝(x_pos)에 바로 텍스트를 박습니다.
    ax.text(x_pos, y_pos, display_text, 
            va='center', ha='left', 
            fontsize=14, 
            fontweight='bold' if "LC909067" in orig_name else 'normal')

ax.set_xlim(0, x_limit * 1.8) 
ax.set_ylim(0, len(terms) + 2)
ax.set_axis_off() # 축 숫자 빠잉 

plt.rc('font', size=14) # 내부 글꼴 사이즈
plt.rc('axes', titlesize=20) # 제모옥은 이 크기로 하겠습니다 
plt.title("Influenza A (H3N2) HA Phylogenetic Tree by Region/Year")
plt.tight_layout()
plt.savefig("Influenza_H3N2_Final_Tree.png", dpi=300, bbox_inches='tight')
plt.xlabel("Genetic Distance (Substitutions per site)")
plt.show()

In [ ]:
# 사람을 공격하는 바이러스만 찾습니다 
human_terms = [t for t in tree.get_terminals() if 'canine' not in str(t.name).lower()]

# 유전적 거리순으로 정렬 
human_distances = [(tree.distance(t), t.name) for t in human_terms]
human_distances.sort(key=lambda x: x[0], reverse=True)

print("=== 🚨 독감 변종 TOP 5 ===")
print("-" * 70)
print(f"{'순위':<4} | {'ID':<12} | {'변이도':<8} | {'신상 정보'}")
print("-" * 70)

for i, (dist, name) in enumerate(human_distances[:5], 1):
    target_id = str(name)
    found_info = "정보 없음"
    
    # alignment 데이터에서 상세 지역/연도 정보 매칭
    for record in alignment:
        if target_id in record.description or target_id in record.id:
            full_info = record.description if record.description else record.id
            if '_' in full_info:
                parts = full_info.split('_')
                found_info = f"[{parts[0]}] {parts[1].split(' ')[0]}"
            else:
                # description에서 연도/지역 추출 시도 (괄호 안 정보 등)
                found_info = full_info.split('virus (')[1].split(')')[0] if '(' in full_info else full_info
            break
            
    print(f"{i:<5} | {target_id:<12} | {dist:.4f} | {found_info}")

print("-" * 70)
print("※ 변이도가 높을수록 기존 면역 체계를 회피할 가능성이 큽니다.")